# Country-Borders Geometry — Reproduction

Minimal, self-contained reproduction of the headline result: **Llama-3.1-8B encodes European country relations as a single, statistically-significant, _specifically geographic_ 2-D map**, recovered from a purely _relational_ directional-border task ("Which country lies to the east of Spain?").

Pipeline: baseline → subspace → per-answer-country centroids → geographic-isomorphism regression → permutation/bootstrap null + geography-residual probe. Runs on a single GPU (RunPod-class) in minutes. Full interpretation: `result/REPORT.md`. Robustness logic is imported from `code/analyses/geometry_robustness.py` (not duplicated).

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
from IPython.display import Image, display

# RunPod convenience: make `uv` visible to the kernel's subprocess.
if shutil.which("uv") is None:
    for d in ("/root/.local/bin", os.path.expanduser("~/.local/bin"),
              "/root/.cargo/bin", "/usr/local/bin"):
        if os.path.exists(os.path.join(d, "uv")):
            os.environ["PATH"] = d + os.pathsep + os.environ["PATH"]; break
print("uv ->", shutil.which("uv"))

import causalab
PROJECT_ROOT   = Path(causalab.__file__).resolve().parent.parent
SESSION_NAME   = (PROJECT_ROOT / "agent_logs" / ".current").read_text().strip()
SESSION_DIR    = PROJECT_ROOT / "agent_logs" / SESSION_NAME
RUNNERS_DIR    = SESSION_DIR / "code" / "configs" / "runners" / "country_borders"
ARTIFACTS_ROOT = SESSION_DIR / "artifacts" / "country_borders" / "llama31_8b"
RESULT_DIR     = SESSION_DIR / "result"
RUNNERS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

def run_session(name: str, yaml_str: str) -> int:
    (RUNNERS_DIR / f"{name}.yaml").write_text(yaml_str)
    cmd = ["bash", "scripts/run_exp.sh", "--experiment-root", str(ARTIFACTS_ROOT), name]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT).returncode

print("session:", SESSION_NAME)

## 1. Baseline

Confirm the model solves the directional-border task. Strict accuracy is reported over the **584 geographically valid** prompts; the 960 enumerated total includes 376 structurally-empty cells that can never match (REPORT §6.1).

In [ ]:
BASELINE_YAML = """# @package _global_
defaults:
  - /base
  - /task: country_borders
  - /model: llama31_8b
  - /analysis/baseline
  - _self_
task: {enumerate_all: true, target_variable: country}
baseline: {visualization: {figure_format: png}}
"""
assert run_session("country_borders_baseline", BASELINE_YAML) == 0, "baseline failed"
acc = json.loads((ARTIFACTS_ROOT / "baseline" / "accuracy.json").read_text())
print(acc)
print(f"strict accuracy on valid prompts: {acc['correct']}/584 = {acc['correct']/584:.1%}")

## 2. Subspace

PCA-32 features at layer 28, last-token position. Layer 28 is a deliberate carry-forward (REPORT §9); completing `locate` to select it empirically is a tracked next step (REPORT §10).

In [ ]:
LAYER, TOKEN_POSITION = 28, "last_token"
SUBSPACE_YAML = f"""# @package _global_
defaults:
  - /base
  - /task: country_borders
  - /model: llama31_8b
  - /analysis/subspace
  - _self_
task: {{enumerate_all: true, target_variable: country}}
subspace:
  method: pca
  layers: [{LAYER}]
  k_features: 32
  token_positions: [{TOKEN_POSITION}]
  visualization: {{figure_format: png}}
"""
assert run_session("country_borders_subspace", SUBSPACE_YAML) == 0, "subspace failed"

## 3. Per-answer-country centroids & geographic isomorphism

One centroid per **answer** country = mean of answer-position activations over all (entity, direction) prompts that resolve to it (entity-disentangled). Regress capital (lat, lon) onto the centroid PCs.

In [ ]:
sys.path.insert(0, str(SESSION_DIR / "code" / "analyses"))
import importlib, geometry_robustness as gr
importlib.reload(gr)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from causalab.tasks.country_borders import LAT_LON_OF

SUBSPACE_ROOT = ARTIFACTS_ROOT / "subspace" / "pca_k32" / "country"
valid_countries, centroids_v, n_examples = gr.build_centroids(SUBSPACE_ROOT)
pca = PCA(n_components=3).fit(centroids_v)
P   = pca.transform(centroids_v)
evr = pca.explained_variance_ratio_
lats = np.array([LAT_LON_OF[c][0] for c in valid_countries])
lons = np.array([LAT_LON_OF[c][1] for c in valid_countries])
geo  = np.column_stack([lats, lons])

print(f"{len(valid_countries)} centroids x {centroids_v.shape[1]} dims  ({n_examples} valid examples)\n")
for k in range(3):
    r = LinearRegression().fit(geo, P[:, k])
    print(f"PC{k}: R^2={r.score(geo, P[:, k]):.3f}  "
          f"beta_lat={r.coef_[0]:+.3f}  beta_lon={r.coef_[1]:+.3f}  (var {evr[k]:.1%})")

# Best in-plane rotation of the PC0/PC1 plane onto true (lat, lon).
theta, combined_r2 = gr.best_rotation(P[:, :2], geo)
Rm = np.array([[np.cos(theta), -np.sin(theta)],
               [np.sin(theta),  np.cos(theta)]])
P_rot = P[:, :2] @ Rm
print(f"\nbest in-plane rotation {np.degrees(theta):.1f} deg  ->  "
      f"combined R^2 = {combined_r2:.3f} / 2.0")

# --- Cosmetic alignment to true compass orientation (DISPLAY ONLY) ----------
# The optimiser searches rotations only (not reflections) and PCA axis signs
# are non-deterministic, so the recovered plane can sit flipped/rotated vs
# true north. R^2 above is orientation-invariant and is NOT affected by this.
# Requested transform: rotate 90 deg clockwise, then mirror across the
# y-axis  ->  (x, y) -> (-y, -x).  If a future PCA run flips a sign and the
# map looks mirrored, negate one of the two lines below.
disp_x = -P_rot[:, 1]
disp_y = -P_rot[:, 0]

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].scatter(lons, lats, s=80, c="steelblue", edgecolors="black", linewidths=0.5)
for c, la, lo in zip(valid_countries, lats, lons):
    ax[0].annotate(c, (lo, la), fontsize=7, alpha=0.7)
ax[0].set(xlabel="Longitude (deg E)", ylabel="Latitude (deg N)",
          title="Actual geography (capital coords)")
ax[0].grid(alpha=0.3)
ax[1].scatter(disp_x, disp_y, s=80, c="steelblue",
              edgecolors="black", linewidths=0.5)
for c, x, y in zip(valid_countries, disp_x, disp_y):
    ax[1].annotate(c, (x, y), fontsize=7, alpha=0.7)
ax[1].set(xlabel="PC1 (aligned)", ylabel="PC0 (aligned)",
          title=f"Centroid PCA, rotated {np.degrees(theta):.0f} deg + compass-aligned "
                f"(combined R^2={combined_r2:.2f})")
ax[1].grid(alpha=0.3)
plt.tight_layout()
(RESULT_DIR / "figures").mkdir(parents=True, exist_ok=True)
plt.savefig(RESULT_DIR / "figures" / "pca_vs_geography.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 4. Robustness — significance & specificity

Permutation null + bootstrap CI for the combined R², and a geography-residual probe: regress (lat, lon) out of the 32-d centroids and re-probe. If East/West and linguistic family are geographic shadows, their lift collapses to chance.

In [ ]:
iso = gr.geographic_isomorphism_null(valid_countries, centroids_v,
                                     n_perm=2000, n_boot=2000, seed=0)
null_samples = iso.pop("_null_samples")
print("combined R^2     :", iso["observed_combined_r2"], "/ 2.0")
print("permutation p    :", iso["permutation"]["p_value"],
      " (null max", iso["permutation"]["null_max"], ")")
print("bootstrap 95% CI :", iso["bootstrap"]["ci95"])

rp = gr.residual_probe(valid_countries, centroids_v)
for axis, d in rp.items():
    print(f"{axis:18s}: raw lift {d['raw']['lift']:+.3f}  ->  "
          f"residual {d['residual']['lift']:+.3f}")

fig = RESULT_DIR / "figures" / "geometry_null_hist.png"
if fig.exists():
    display(Image(filename=str(fig)))

## 5. Finding

Llama-3.1-8B encodes European country relations as a single, globally coherent ~2-D **geographic** map: PC0 ≈ latitude (R² 0.77), PC1 ≈ longitude (R² 0.83), recovered natively (best in-plane rotation only −11.7°). It is **statistically significant** (permutation p = 0.0005; bootstrap 95% CI [1.24, 1.80] entirely above null max 0.55) and **specifically geographic**: East/West and linguistic family collapse to chance once geography is partialled out, and EU membership is not encoded at all. The map is recovered from a purely _relational_ task.

Open (REPORT §10): the retrieval-vs-computation contrast (the decisive novelty vs. Gurnee & Tegmark, arXiv 2310.02207) and completing `locate`. Full interpretation: `result/REPORT.md`.